# BookMatch: гибридная система

## 1. Признаки

Статистики оценок и профиль пользователя строятся только по train. Активность — число оценок, дат событий нет. Разброс оценок — стандартное отклонение, для одной оценки принимается равным нулю.

Тематика описывается 20 частыми тегами без служебных меток. Названия и теги используются также для сходства книги с понравившейся историей. Это метаданные каталога, а не статистики validation или test.

Для неизвестного пользователя используются средняя train, нулевая активность и нулевое сходство. Для книги без оценок сохраняются её контентные признаки. Пользу признаков проверим при сравнении моделей.

In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.data import load_data, split_ratings
from src.features import HybridFeatures

books, ratings, tags, book_tags = load_data()
train, validation, test = split_ratings(ratings)
features = HybridFeatures().fit(train, books, tags, book_tags)
display(features.users.head())
display(features.books.head())

,user_mean,user_count,user_std
user_id,,,
1,4.000000,1,0.00000
2,5.000000,1,0.00000
3,1.000000,2,0.00000
4,5.000000,1,0.00000
5,4.666667,3,0.57735


,book_mean,book_count,book_std,tag_fiction,tag_i-own,tag_adult,tag_novels,tag_own-it,tag_contemporary,tag_read-in-2015,...,tag_books,tag_adult-fiction,tag_read-in-2013,tag_book-club,tag_audible,tag_fantasy,tag_romance,tag_audio-books,tag_abandoned,tag_novel
book_id,,,,,,,,,,,,,,,,,,,,,
1,4.232323,99,0.890176,1,1,0,1,1,1,0,...,1,0,1,1,0,1,1,0,0,1
2,4.210000,100,1.037625,1,1,0,1,1,1,1,...,1,0,1,0,0,1,0,0,0,1
3,3.070707,99,1.465609,1,1,0,1,1,1,0,...,1,0,0,1,0,1,0,0,1,0
4,4.460000,100,0.833879,1,1,1,1,1,1,1,...,1,1,0,1,0,0,0,0,0,1
5,3.887755,98,1.082977,1,1,1,1,1,0,1,...,1,1,1,1,0,0,1,0,0,1


In [2]:
validation_features = features.transform(validation)
display(validation_features.head())
print('Пар:', len(validation_features))
print('Признаков:', validation_features.shape[1] - 2)
print('Пропусков:', validation_features.isna().sum().sum())

,user_id,book_id,user_mean,user_count,user_std,book_mean,book_count,book_std,tag_fiction,tag_i-own,...,tag_adult-fiction,tag_read-in-2013,tag_book-club,tag_audible,tag_fantasy,tag_romance,tag_audio-books,tag_abandoned,tag_novel,history_similarity
0,30034,5534,3.302326,43,0.464701,3.416667,84,0.996480,1,1,...,1,1,1,0,0,1,0,1,0,0.174594
1,5436,693,3.668508,181,1.174049,3.402062,97,0.996450,1,1,...,1,0,1,1,1,0,0,0,1,0.465910
2,10111,967,4.021858,183,0.999760,3.707071,99,0.823786,1,1,...,1,0,0,0,1,0,0,1,1,0.253147
3,7841,4385,3.375000,32,0.707107,4.043011,93,0.858639,1,0,...,1,1,1,1,0,0,1,1,1,0.083441
4,13089,5385,3.444444,72,1.046643,3.125000,96,1.068693,1,1,...,1,0,1,1,0,1,0,1,1,0.450207


Пар: 45054
Признаков: 27
Пропусков: 0


ID служат ключами, оценка не входит в признаки. При обучении ранжировщика признаки нужно строить на внутренней обучающей части, а целевые пары брать из отложенной части. Иначе оценка взаимодействия попадёт в его собственные признаки.